# Network Centrality Analysis (מתח"מים Validation Study)

Builds the planned mass-transit network graph from **per-line polyline shapefiles** + the nodes-on-lines CSV, contracts stations into hubs, computes **weighted betweenness centrality** (primary), closeness and degree, and validates them against the Monte Carlo hub rankings.

**Why betweenness?** The framework defines a מתח"מ as the "operational heart" of the network — betweenness measures exactly that: the share of shortest paths passing *through* a node (transfer/interchange importance). None of the 5 scoring criteria capture it.

**Before running**: execute `python scripts/inspect_transit_line_shapefiles.py` to verify shapefile↔LINE_ID matching and direction-pair node sharing.

Pipeline: `SHP polylines → stop ordering by projection → station graph (L-space) → hub contraction → centrality → validation vs rankings`


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # project root

import pandas as pd

from src.config import (
    TRANSIT_LINES_DIR, LINE_SHAPEFILE_MAPPING_CSV,
    CENTRALITY_MODES, CENTRALITY_EDGE_WEIGHT, RESULTS_DIR,
)
from src.data.loaders import load_transit_nodes, load_lines_and_modes
from src.network import (
    load_line_geometries, match_lines_to_shapefiles,
    build_station_graph, build_node_to_hub_mapping, contract_to_hub_graph,
    compute_centrality_metrics, normalize_centrality,
    compare_with_rankings, find_divergent_hubs, write_validation_report,
)
from src.network.validation import plot_centrality_vs_score, create_centrality_map


## 1. Configuration — set your input paths


In [ ]:
NODES_CSV = Path('../data/raw/All_nodes+lines.csv')  # node, LINE_ID, X/Y (EPSG:2039)
LINES_MODES_CSV = Path('../data/raw/Lines_and_Planned_Mode.csv')
SCORED_HUBS_CSV = Path('../data/results/scored_hubs_final.csv')
LINES_DIR = TRANSIT_LINES_DIR  # data/raw/transit_lines/

WEIGHT = CENTRALITY_EDGE_WEIGHT  # 'time_min' or 'length_m'
MODES = set(CENTRALITY_MODES)


## 2. Load inputs and match shapefiles to lines


In [ ]:
nodes_gdf = load_transit_nodes(NODES_CSV)
lines_modes = load_lines_and_modes(LINES_MODES_CSV)
scored_hubs = pd.read_csv(SCORED_HUBS_CSV, encoding='utf-8-sig')

manual = (pd.read_csv(LINE_SHAPEFILE_MAPPING_CSV)
          if LINE_SHAPEFILE_MAPPING_CSV.exists() else None)

mode_by_line = lines_modes.set_index(
    lines_modes['Line_ModelName'].astype(str))['Mode_Planned'].to_dict()
in_scope = {l for l in nodes_gdf['LINE_ID'].astype(str).unique()
            if mode_by_line.get(l) in MODES}

line_geoms = load_line_geometries(LINES_DIR)
geom_by_line, match_report = match_lines_to_shapefiles(
    in_scope, line_geoms, manual_mapping=manual)
match_report['matched_by'].value_counts()


## 3. Build the station graph and contract into hubs

Stops are ordered by projecting them onto each line's polyline (`line.project`). Check `qa_df` for skipped lines, projection warnings and loop flags **before trusting the results** — a missing central line biases betweenness for the whole network.


In [ ]:
G_stations, qa_df = build_station_graph(
    nodes_gdf, lines_modes, geom_by_line, modes=MODES)
qa_df[~qa_df['status'].str.startswith('ok')]  # lines needing attention


In [ ]:
node_to_hub = build_node_to_hub_mapping(scored_hubs)
G_hubs = contract_to_hub_graph(G_stations, node_to_hub)


## 4. Compute centrality


In [ ]:
centrality = normalize_centrality(
    compute_centrality_metrics(G_hubs, weight=WEIGHT))
centrality.sort_values('betweenness', ascending=False).head(15)


## 5. Validate against Monte Carlo rankings


In [ ]:
results = compare_with_rankings(centrality, scored_hubs)
results['correlations']


In [ ]:
divergent = find_divergent_hubs(results['eligible'], results['rank_col'])
divergent  # positive rank_diff = network heart the MC ranking underrates


In [ ]:
results['criterion_correlations']
# Expected: degree <-> service score high (construction cross-check);
# closeness <-> location score negative (location boosts periphery)


In [ ]:
top_stations = (centrality[~centrality['is_hub']]
                .nlargest(20, 'betweenness')
                [['betweenness', 'closeness', 'degree', 'n_lines', 'n_modes']])
top_stations  # structural hearts not selected as hubs


## 6. Outputs (report, scatter, interactive map)


In [ ]:
write_validation_report(results, divergent, qa_df, out_dir=RESULTS_DIR,
                        weight=WEIGHT, top_stations=top_stations)
plot_centrality_vs_score(results, RESULTS_DIR / 'centrality_vs_ranking_scatter.png')
create_centrality_map(centrality, RESULTS_DIR / 'network_centrality_map.html',
                      line_geometries=geom_by_line, merged=results['merged'])


## 7. Sensitivity runs

How robust is the headline correlation to the modeling choices?


In [ ]:
# 7a. Distance weights instead of generalized time
cent_dist = normalize_centrality(
    compute_centrality_metrics(G_hubs, weight='length_m'))
res_dist = compare_with_rankings(cent_dist, scored_hubs)
print('time_min :', results['correlations']['betweenness_vs_score'])
print('length_m :', res_dist['correlations']['betweenness_vs_score'])


In [ ]:
# 7b. Without BRT (does the dense/slow BRT layer dilute rail centrality?)
modes_no_brt = MODES - {'BRT'}
G_nb, _ = build_station_graph(nodes_gdf, lines_modes, geom_by_line, modes=modes_no_brt)
H_nb = contract_to_hub_graph(G_nb, node_to_hub)
res_nb = compare_with_rankings(
    normalize_centrality(compute_centrality_metrics(H_nb, weight=WEIGHT)),
    scored_hubs)
print('no BRT   :', res_nb['correlations']['betweenness_vs_score'])


In [ ]:
# 7c. Station-level (uncontracted) betweenness, aggregated per hub by max
# Contrast with the contracted-graph result: contraction favours large
# multi-station hubs (free internal transfers), max-aggregation does not.
cent_stations = compute_centrality_metrics(G_stations, weight=WEIGHT)
hub_max = {}
for node, hub in node_to_hub.items():
    if node in cent_stations.index:
        b = cent_stations.loc[node, 'betweenness']
        hub_max[hub] = max(hub_max.get(hub, 0.0), b)
station_level = pd.Series(hub_max, name='betweenness_station_max')
comparison = centrality.loc[centrality['is_hub'], ['betweenness']].join(station_level)
comparison.corr(method='spearman')
